# Hotel Booking Cancellation Prediction

## Notebook 03 — Stage 6: Model Development and Comparison

**Module:** IT3051 – Fundamentals of Data Mining  
**Group:** Group 27 – Entropy Zero  
**Task:** Supervised binary classification  
**Target:** `is_canceled`

- `0` — Not Cancelled
- `1` — Cancelled

### Stage 6 objective

This notebook implements the finalized Stage 6 methodology and compares four baseline classification algorithms using **training data only**:

1. Logistic Regression
2. Decision Tree
3. Random Forest
4. XGBoost

The purpose is to establish fair, reproducible baseline evidence before Stage 7 optimization.

### Final-test lock

Notebook 02 created a chronological final holdout containing the later bookings. **That final test set is intentionally not loaded anywhere in this notebook.**

Stage 6 uses only:

- `X_train_engineered.csv`
- `y_train.csv`

The final chronological test set must not influence:

- model comparison;
- preprocessing decisions;
- feature selection;
- hyperparameter tuning;
- class-weight decisions;
- threshold selection;
- or Stage 7 shortlisting.

No model is assumed to be best before the validation experiments are executed.

## 1. Reproducibility and Environment

The project uses `RANDOM_STATE = 42` for estimators that support a random seed.

Notebook 03 reuses the same repository structure as the earlier notebooks. In Google Colab, the repository is cloned only if it is not already present.

XGBoost is the only dependency used here that is not part of the finalized Stage 4 `scikit-learn` stack. The minimal installation command is included below.

> If XGBoost remains part of the submitted modelling workflow, add `xgboost` to `requirements.txt` and preferably pin the tested version after successful execution.

In [ ]:
%pip install -q xgboost

In [ ]:
import os
import time
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sklearn
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

import xgboost
from xgboost import XGBClassifier

RANDOM_STATE = 42

print('Python:', platform.python_version())
print('pandas:', pd.__version__)
print('NumPy:', np.__version__)
print('scikit-learn:', sklearn.__version__)
print('XGBoost:', xgboost.__version__)
print('Random state:', RANDOM_STATE)

### Connect to the GitHub repository

The processed CSVs produced by Notebook 02 are reproducible generated artifacts. They may not exist in a fresh clone.

If this notebook is opened in a new Colab runtime, run Notebook 02 first in the same runtime, or otherwise place its generated Stage 4 training artifacts in `data/processed/`.

In [ ]:
import subprocess

REPO_NAME = 'IT3051-Group27-Hotel-Cancellation-Prediction'
REPO_URL = f'https://github.com/sudulran/{REPO_NAME}.git'
REPO_PATH = Path(f'/content/{REPO_NAME}')

if not REPO_PATH.exists():
    subprocess.run(
        ['git', 'clone', REPO_URL, str(REPO_PATH)],
        check=True,
    )
else:
    print('Repository already present in this runtime.')

os.chdir(REPO_PATH)
print('Working directory:', Path.cwd())

## 2. Load Only the Finalized Training Data

Stage 6 deliberately loads only the engineered, **unencoded** training predictors and target created by Notebook 02.

The already transformed matrix from Notebook 02 is **not** used for cross-validation. Learned preprocessing must be refitted separately inside each cross-validation training fold.

In [ ]:
X_TRAIN_PATH = Path('data/processed/X_train_engineered.csv')
Y_TRAIN_PATH = Path('data/processed/y_train.csv')

required_files = [X_TRAIN_PATH, Y_TRAIN_PATH]
missing_files = [str(path) for path in required_files if not path.exists()]

if missing_files:
    raise FileNotFoundError(
        'Required Stage 4 training artifacts are missing:\n'
        + '\n'.join(f' - {path}' for path in missing_files)
        + '\n\nRun Notebook 02 — Preprocessing and Feature Engineering first. '
          'Notebook 02 must generate X_train_engineered.csv and y_train.csv '
          'under data/processed/. The final test files are not required and '
          'must not be loaded for Stage 6.'
    )

X_train = pd.read_csv(X_TRAIN_PATH)
y_train_frame = pd.read_csv(Y_TRAIN_PATH)

TARGET = 'is_canceled'

if list(y_train_frame.columns) != [TARGET]:
    raise ValueError(
        f"Expected y_train.csv to contain exactly one column named '{TARGET}', "
        f'but found: {list(y_train_frame.columns)}'
    )

y_train = y_train_frame[TARGET].copy()

# CSV reading can infer the engineered booking_month as an integer.
# Restore it to the categorical/string representation finalized in Notebook 02.
X_train['booking_month'] = X_train['booking_month'].astype(str)

# Use clean positional indices for the shared cross-validation folds.
X_train = X_train.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)

print('Training predictors loaded:', X_train.shape)
print('Training target loaded:', y_train.shape)
print('Final test data loaded in Stage 6: NO')

## 3. Training-Data Integrity Checks

Notebook 03 treats the finalized Stage 4 output as a contract. It should fail loudly rather than silently continue if the expected rows, predictors, target, leakage exclusions, or missing-value pattern have changed.

Finalized Stage 4 expectations:

- **95,401** training observations;
- **32** predictors before encoding;
- target values only `0` and `1`;
- raw/future/leakage/helper fields excluded;
- only `children` and `country` retain missing values before the model pipeline;
- `children` has 4 missing training values;
- `country` has 415 missing training values.

In [ ]:
EXPECTED_N_ROWS = 95_401

EXPECTED_PREDICTORS = [
    'hotel',
    'lead_time',
    'arrival_date_year',
    'arrival_date_month',
    'arrival_date_week_number',
    'arrival_date_day_of_month',
    'stays_in_weekend_nights',
    'stays_in_week_nights',
    'adults',
    'children',
    'babies',
    'meal',
    'country',
    'market_segment',
    'distribution_channel',
    'is_repeated_guest',
    'previous_cancellations',
    'previous_bookings_not_canceled',
    'reserved_room_type',
    'deposit_type',
    'customer_type',
    'adr',
    'required_car_parking_spaces',
    'total_of_special_requests',
    'total_nights',
    'total_guests',
    'has_children',
    'has_agent',
    'has_company',
    'is_zero_guest',
    'is_zero_night',
    'booking_month',
]

FORBIDDEN_COLUMNS = {
    'is_canceled',
    'reservation_status',
    'reservation_status_date',
    'assigned_room_type',
    'booking_changes',
    'days_in_waiting_list',
    'agent',
    'company',
    'arrival_date',
    'booking_date',
}

assert len(X_train) == EXPECTED_N_ROWS, (
    f'Expected {EXPECTED_N_ROWS:,} training rows, found {len(X_train):,}.'
)
assert len(y_train) == EXPECTED_N_ROWS, (
    f'Expected {EXPECTED_N_ROWS:,} target rows, found {len(y_train):,}.'
)
assert X_train.shape[1] == 32, (
    f'Expected 32 predictors, found {X_train.shape[1]}.'
)
assert list(X_train.columns) == EXPECTED_PREDICTORS, (
    'Predictor columns/order do not match the finalized Notebook 02 contract.'
)
assert X_train.index.equals(y_train.index), (
    'X_train and y_train row indices are not aligned.'
)
assert set(y_train.dropna().unique()) == {0, 1}, (
    f'Target must contain only 0 and 1. Found: {sorted(y_train.dropna().unique())}'
)
assert not y_train.isna().any(), 'Target contains missing values.'

forbidden_present = sorted(FORBIDDEN_COLUMNS.intersection(X_train.columns))
assert not forbidden_present, (
    f'Forbidden leakage/timing/identifier/helper columns found: {forbidden_present}'
)

missing_counts = X_train.isna().sum()
nonzero_missing = missing_counts[missing_counts > 0]

expected_missing_counts = pd.Series(
    {'children': 4, 'country': 415},
    dtype='int64',
)

assert set(nonzero_missing.index) == set(expected_missing_counts.index), (
    'Unexpected missing-value columns found. '
    f'Observed: {nonzero_missing.to_dict()}'
)
for column, expected_count in expected_missing_counts.items():
    assert int(missing_counts[column]) == expected_count, (
        f"Expected {expected_count} missing values in '{column}', "
        f'found {int(missing_counts[column])}.'
    )

assert X_train['booking_month'].dtype == object, (
    'booking_month should be restored as a categorical/string column.'
)

# Demonstrate that final-test variables are not part of this runtime state.
assert 'X_test' not in globals()
assert 'y_test' not in globals()

print('All Stage 4 training-data integrity checks passed.')
print('\nObserved pre-pipeline missing values:')
display(nonzero_missing.to_frame('Missing Count'))

## 4. Reconstruct `booking_date` for Cross-Validation Only

The engineered CSV intentionally does not contain `booking_date`.

For chronological validation, it is reconstructed from variables that remain in the training data:

**Arrival Date = arrival year + arrival month + arrival day**

**Booking Date = Arrival Date − lead_time days**

The reconstructed date is used **only** for:

- chronological ordering;
- constructing validation folds;
- verifying temporal separation.

It is **not** added to `X_train` and is never given to a classifier as a predictive feature.

This preserves the Stage 4 feature set while still allowing realistic time-aware validation.

In [ ]:
arrival_date_for_cv = pd.to_datetime(
    X_train['arrival_date_year'].astype(str)
    + '-'
    + X_train['arrival_date_month'].astype(str)
    + '-'
    + X_train['arrival_date_day_of_month'].astype(str),
    format='%Y-%B-%d',
    errors='raise',
)

booking_date_for_cv = (
    arrival_date_for_cv
    - pd.to_timedelta(X_train['lead_time'], unit='D')
)

assert booking_date_for_cv.notna().all()
assert len(booking_date_for_cv) == len(X_train)
assert 'booking_date' not in X_train.columns

print('Reconstructed training booking-date range:')
print(booking_date_for_cv.min(), 'to', booking_date_for_cv.max())

print('\nbooking_date is a predictor:', 'booking_date' in X_train.columns)

## 5. Recreate the Finalized Stage 4 Preprocessor

The same finalized feature groups and transformations from Notebook 02 are recreated here.

### General numerical features
- `SimpleImputer(strategy="median")`
- `RobustScaler()`

### `children`
- `SimpleImputer(strategy="most_frequent")`
- `RobustScaler()`

### Binary indicators
- passthrough as `0/1`

### Categorical features
- `SimpleImputer(strategy="constant", fill_value="Unknown")`
- `OneHotEncoder(handle_unknown="ignore")`

The function below **returns a fresh unfitted preprocessor**. It is not fitted before cross-validation.

In [ ]:
CATEGORICAL_COLUMNS = [
    'hotel',
    'arrival_date_month',
    'meal',
    'country',
    'market_segment',
    'distribution_channel',
    'reserved_room_type',
    'deposit_type',
    'customer_type',
    'booking_month',
]

BINARY_COLUMNS = [
    'is_repeated_guest',
    'has_children',
    'has_agent',
    'has_company',
    'is_zero_guest',
    'is_zero_night',
]

NUMERIC_COLUMNS_TO_SCALE = [
    'lead_time',
    'arrival_date_year',
    'arrival_date_week_number',
    'arrival_date_day_of_month',
    'stays_in_weekend_nights',
    'stays_in_week_nights',
    'adults',
    'babies',
    'previous_cancellations',
    'previous_bookings_not_canceled',
    'adr',
    'required_car_parking_spaces',
    'total_of_special_requests',
    'total_nights',
    'total_guests',
]

CHILDREN_COLUMN = ['children']

all_grouped_columns = (
    NUMERIC_COLUMNS_TO_SCALE
    + CHILDREN_COLUMN
    + BINARY_COLUMNS
    + CATEGORICAL_COLUMNS
)

assert len(all_grouped_columns) == 32
assert set(all_grouped_columns) == set(EXPECTED_PREDICTORS)
assert len(all_grouped_columns) == len(set(all_grouped_columns))

def make_finalized_preprocessor():
    """Return a fresh, unfitted copy of the finalized Notebook 02 preprocessor."""
    numeric_pipeline = Pipeline(
        steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', RobustScaler()),
        ]
    )

    children_pipeline = Pipeline(
        steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('scaler', RobustScaler()),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                'imputer',
                SimpleImputer(
                    strategy='constant',
                    fill_value='Unknown',
                ),
            ),
            (
                'encoder',
                OneHotEncoder(handle_unknown='ignore'),
            ),
        ]
    )

    return ColumnTransformer(
        transformers=[
            ('numeric', numeric_pipeline, NUMERIC_COLUMNS_TO_SCALE),
            ('children', children_pipeline, CHILDREN_COLUMN),
            ('binary', 'passthrough', BINARY_COLUMNS),
            ('categorical', categorical_pipeline, CATEGORICAL_COLUMNS),
        ]
    )

preprocessor_check = make_finalized_preprocessor()
print(preprocessor_check)
print('\nPreprocessor has NOT been fitted.')

## 6. Five-Fold Grouped Expanding-Window Chronological Validation

A shuffled split would allow future bookings to influence a model that is validated on earlier bookings. That does not match the intended future-booking prediction scenario.

This notebook therefore uses five expanding-window folds:

- Fold 1: earliest ~50% train, next ~10% validate
- Fold 2: earliest ~60% train, next ~10% validate
- Fold 3: earliest ~70% train, next ~10% validate
- Fold 4: earliest ~80% train, next ~10% validate
- Fold 5: earliest ~90% train, final ~10% validate

### Complete-date grouping rule

Records sharing the same reconstructed `booking_date` are kept together. A boundary is selected only after a complete date group, so the same booking date can never appear in both a fold's training and validation portions.

Every fold must satisfy:

`max(training booking_date) < min(validation booking_date)`

In [ ]:
def build_grouped_expanding_time_folds(
    booking_dates,
    initial_train_fraction=0.50,
    validation_fraction=0.10,
    n_splits=5,
):
    """
    Build expanding chronological folds using complete booking-date groups.

    Boundaries are chosen using cumulative row proportions by unique date.
    For each target proportion, the latest complete date whose cumulative
    proportion does not exceed the target is used. The final endpoint uses
    the maximum available booking date.
    """
    booking_dates = pd.Series(booking_dates).reset_index(drop=True)

    if booking_dates.isna().any():
        raise ValueError('booking_dates contains missing values.')

    expected_final_fraction = (
        initial_train_fraction + n_splits * validation_fraction
    )
    if not np.isclose(expected_final_fraction, 1.0):
        raise ValueError(
            'Fractions must cover the full training set. '
            f'Received final cumulative fraction {expected_final_fraction:.3f}.'
        )

    date_counts = booking_dates.value_counts().sort_index()
    cumulative_fraction = date_counts.cumsum() / len(booking_dates)

    target_end_fractions = [
        initial_train_fraction + i * validation_fraction
        for i in range(n_splits + 1)
    ]

    boundaries = []
    for target_fraction in target_end_fractions:
        if np.isclose(target_fraction, 1.0):
            boundary = date_counts.index[-1]
        else:
            eligible = cumulative_fraction[
                cumulative_fraction <= target_fraction
            ]
            if eligible.empty:
                raise ValueError(
                    f'No complete booking-date group fits target '
                    f'fraction {target_fraction:.2f}.'
                )
            boundary = eligible.index[-1]
        boundaries.append(boundary)

    if len(set(boundaries)) != len(boundaries):
        raise ValueError(
            'One or more fold boundaries collapsed onto the same date. '
            'Review the date-group distribution.'
        )

    folds = []
    for fold_number in range(1, n_splits + 1):
        train_end = boundaries[fold_number - 1]
        validation_end = boundaries[fold_number]

        train_mask = booking_dates <= train_end
        validation_mask = (
            (booking_dates > train_end)
            & (booking_dates <= validation_end)
        )

        train_idx = np.flatnonzero(train_mask.to_numpy())
        validation_idx = np.flatnonzero(validation_mask.to_numpy())

        if len(train_idx) == 0 or len(validation_idx) == 0:
            raise ValueError(
                f'Fold {fold_number} contains an empty train or validation set.'
            )

        train_dates = booking_dates.iloc[train_idx]
        validation_dates = booking_dates.iloc[validation_idx]

        assert train_dates.max() < validation_dates.min(), (
            f'Fold {fold_number} violates chronological separation.'
        )
        assert set(train_dates.unique()).isdisjoint(
            set(validation_dates.unique())
        ), f'Fold {fold_number} splits a booking-date group.'

        folds.append((train_idx, validation_idx))

    return folds, boundaries


time_folds, fold_boundaries = build_grouped_expanding_time_folds(
    booking_date_for_cv,
    initial_train_fraction=0.50,
    validation_fraction=0.10,
    n_splits=5,
)

print('Created', len(time_folds), 'grouped chronological folds.')
print('Boundary dates:')
for i, boundary in enumerate(fold_boundaries):
    label = 'Initial train end' if i == 0 else f'Validation {i} end'
    print(f' - {label}: {boundary.date()}')

In [ ]:
fold_summary_rows = []

for fold_number, (train_idx, validation_idx) in enumerate(
    time_folds,
    start=1,
):
    train_dates = booking_date_for_cv.iloc[train_idx]
    validation_dates = booking_date_for_cv.iloc[validation_idx]

    assert train_dates.max() < validation_dates.min()
    assert set(train_dates.unique()).isdisjoint(
        set(validation_dates.unique())
    )

    fold_summary_rows.append(
        {
            'Fold': fold_number,
            'Training Start Date': train_dates.min().date(),
            'Training End Date': train_dates.max().date(),
            'Validation Start Date': validation_dates.min().date(),
            'Validation End Date': validation_dates.max().date(),
            'Training Rows': len(train_idx),
            'Validation Rows': len(validation_idx),
            'Training Cancellation %': (
                y_train.iloc[train_idx].mean() * 100
            ),
            'Validation Cancellation %': (
                y_train.iloc[validation_idx].mean() * 100
            ),
        }
    )

fold_summary = pd.DataFrame(fold_summary_rows)

display(
    fold_summary.style.format(
        {
            'Training Cancellation %': '{:.2f}',
            'Validation Cancellation %': '{:.2f}',
        }
    )
)

print(
    '\nAll folds passed the strict temporal-separation check: '
    'max(train date) < min(validation date).'
)

## 7. Evaluation Metrics

### Primary model-selection metric: ROC-AUC

ROC-AUC is the primary Stage 6 metric because it evaluates how well a model ranks cancelled bookings above non-cancelled bookings across possible classification thresholds. It is therefore **threshold-independent** and avoids selecting a model simply because the default `0.5` threshold happens to work well.

### Secondary metrics

- **PR-AUC / Average Precision:** useful because cancellation (`1`) is the minority class and the metric focuses on the quality of positive-class ranking.
- **Precision:** among bookings predicted as cancelled, how many really cancelled?
- **Recall:** among all bookings that cancelled, how many were detected?
- **F1-score:** balances cancellation precision and recall.
- **Accuracy:** reported for completeness, but not sufficient by itself for an imbalanced classification problem.
- **Fit time / prediction time:** provides practical computational evidence.

Aggregated out-of-fold confusion matrices are also generated using only the chronological validation portions.

## 8. Define the Four Untuned Baseline Models

These settings are deliberately **baseline settings**, not optimized settings.

No class weighting is enabled in Stage 6. Class-weight experiments belong to Stage 7 and must use training data only.

In [ ]:
baseline_models = {
    'Logistic Regression': LogisticRegression(
        penalty='l2',
        C=1.0,
        solver='liblinear',
        max_iter=2000,
        class_weight=None,
        random_state=RANDOM_STATE,
    ),

    'Decision Tree': DecisionTreeClassifier(
        criterion='gini',
        class_weight=None,
        random_state=RANDOM_STATE,
    ),

    'Random Forest': RandomForestClassifier(
        n_estimators=250,
        class_weight=None,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),

    'XGBoost': XGBClassifier(
        objective='binary:logistic',
        eval_metric='logloss',
        n_estimators=100,
        learning_rate=0.3,
        max_depth=6,
        tree_method='hist',
        scale_pos_weight=1.0,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
}

for name, model in baseline_models.items():
    print(f'\n{name}')
    print(model)

## 9. Build One Leakage-Safe Pipeline per Model

Every model follows the same structure:

`raw engineered fold data → fresh finalized preprocessor → classifier`

The preprocessor is fitted **inside each CV training fold**.

Fitting preprocessing once before CV would allow validation-fold information to affect:

- imputation values;
- RobustScaler medians and interquartile ranges;
- the learned one-hot-encoding category vocabulary.

That would create validation leakage.

Therefore, the transformed `X_train_processed` matrix from Notebook 02 must never be used for this cross-validation experiment.

In [ ]:
def make_model_pipeline(classifier):
    """Create a fresh leakage-safe preprocessing + classifier pipeline."""
    return Pipeline(
        steps=[
            ('preprocessor', make_finalized_preprocessor()),
            ('classifier', clone(classifier)),
        ]
    )

baseline_pipelines = {
    name: make_model_pipeline(model)
    for name, model in baseline_models.items()
}

for name, pipeline in baseline_pipelines.items():
    print(f'{name}: {list(pipeline.named_steps.keys())}')

## 10. Baseline Time-Aware Cross-Validation

The helper below:

1. uses exactly the same five chronological folds for every model;
2. creates a fresh pipeline for every fold;
3. fits preprocessing and the classifier only on the fold's training rows;
4. predicts the later validation rows;
5. records all required metrics and timing;
6. stores time-aware out-of-fold predictions for the validation portions.

The earliest ~50% of the training set is used only as the initial expanding training window, so it does not receive out-of-fold predictions in this Stage 6 scheme.

In [ ]:
def evaluate_baseline_model(
    model_name,
    classifier,
    X,
    y,
    folds,
    booking_dates,
):
    fold_rows = []
    oof_rows = []

    print('=' * 80)
    print(f'Evaluating: {model_name}')
    print('=' * 80)

    for fold_number, (train_idx, validation_idx) in enumerate(
        folds,
        start=1,
    ):
        pipeline = make_model_pipeline(classifier)

        X_fold_train = X.iloc[train_idx]
        y_fold_train = y.iloc[train_idx]
        X_fold_validation = X.iloc[validation_idx]
        y_fold_validation = y.iloc[validation_idx]

        train_dates = booking_dates.iloc[train_idx]
        validation_dates = booking_dates.iloc[validation_idx]

        assert train_dates.max() < validation_dates.min()

        fit_start = time.perf_counter()
        pipeline.fit(X_fold_train, y_fold_train)
        fit_seconds = time.perf_counter() - fit_start

        score_start = time.perf_counter()
        validation_probability = pipeline.predict_proba(
            X_fold_validation
        )[:, 1]
        validation_prediction = (
            validation_probability >= 0.5
        ).astype(int)
        score_seconds = time.perf_counter() - score_start

        metrics_row = {
            'Model': model_name,
            'Fold': fold_number,
            'ROC-AUC': roc_auc_score(
                y_fold_validation,
                validation_probability,
            ),
            'PR-AUC': average_precision_score(
                y_fold_validation,
                validation_probability,
            ),
            'Accuracy': accuracy_score(
                y_fold_validation,
                validation_prediction,
            ),
            'Precision': precision_score(
                y_fold_validation,
                validation_prediction,
                pos_label=1,
                zero_division=0,
            ),
            'Recall': recall_score(
                y_fold_validation,
                validation_prediction,
                pos_label=1,
                zero_division=0,
            ),
            'F1': f1_score(
                y_fold_validation,
                validation_prediction,
                pos_label=1,
                zero_division=0,
            ),
            'Fit Time Seconds': fit_seconds,
            'Scoring Time Seconds': score_seconds,
            'Training Rows': len(train_idx),
            'Validation Rows': len(validation_idx),
            'Training End Date': train_dates.max().date(),
            'Validation Start Date': validation_dates.min().date(),
            'Validation End Date': validation_dates.max().date(),
        }
        fold_rows.append(metrics_row)

        fold_oof = pd.DataFrame(
            {
                'Model': model_name,
                'Fold': fold_number,
                'Row Index': validation_idx,
                'Booking Date': validation_dates.to_numpy(),
                'Actual': y_fold_validation.to_numpy(),
                'Predicted': validation_prediction,
                'Probability Cancelled': validation_probability,
            }
        )
        oof_rows.append(fold_oof)

        print(
            f"Fold {fold_number}: "
            f"ROC-AUC={metrics_row['ROC-AUC']:.4f}, "
            f"PR-AUC={metrics_row['PR-AUC']:.4f}, "
            f"F1={metrics_row['F1']:.4f}, "
            f"fit={fit_seconds:.2f}s"
        )

    fold_results = pd.DataFrame(fold_rows)
    oof_predictions = pd.concat(
        oof_rows,
        ignore_index=True,
    )

    assert not oof_predictions['Row Index'].duplicated().any()

    return fold_results, oof_predictions

In [ ]:
all_fold_results = []
all_oof_predictions = []

for model_name, classifier in baseline_models.items():
    model_fold_results, model_oof_predictions = evaluate_baseline_model(
        model_name=model_name,
        classifier=classifier,
        X=X_train,
        y=y_train,
        folds=time_folds,
        booking_dates=booking_date_for_cv,
    )

    all_fold_results.append(model_fold_results)
    all_oof_predictions.append(model_oof_predictions)

stage6_fold_results = pd.concat(
    all_fold_results,
    ignore_index=True,
)

stage6_oof_predictions = pd.concat(
    all_oof_predictions,
    ignore_index=True,
)

print('\nBaseline cross-validation completed.')
print('Per-fold result rows:', len(stage6_fold_results))
print('OOF prediction rows:', len(stage6_oof_predictions))

display(stage6_fold_results)

## 11. Aggregated Time-Aware Out-of-Fold Confusion Matrices

For each baseline model, predictions from its five later validation windows are combined into one aggregated out-of-fold confusion matrix.

These are **validation** confusion matrices, not final-test confusion matrices.

The fixed `0.5` threshold is used only for this Stage 6 baseline comparison. Threshold optimization, if justified, belongs to Stage 7 and must use training-only validation evidence.

In [ ]:
FIGURE_DIR = Path('reports/figures')
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

def safe_filename(name):
    return (
        name.lower()
        .replace(' ', '_')
        .replace('-', '_')
    )

confusion_matrices = {}

for model_name in baseline_models:
    model_oof = stage6_oof_predictions[
        stage6_oof_predictions['Model'] == model_name
    ]

    cm = confusion_matrix(
        model_oof['Actual'],
        model_oof['Predicted'],
        labels=[0, 1],
    )
    confusion_matrices[model_name] = cm

    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=['Not Cancelled (0)', 'Cancelled (1)'],
    ).plot(
        ax=ax,
        values_format='d',
        colorbar=False,
    )
    ax.set_title(
        f'{model_name}\nAggregated Time-Aware Validation Confusion Matrix'
    )
    fig.tight_layout()

    output_path = (
        FIGURE_DIR
        / f'stage6_{safe_filename(model_name)}_validation_confusion_matrix.png'
    )
    fig.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.show()

print('Validation confusion-matrix figures saved under reports/figures/.')

## 12. Baseline Comparison Table

The comparison table reports the mean and fold-to-fold standard deviation of the validation metrics.

Values are rounded **only for display**. Full-precision values remain available in the saved result files.

ROC-AUC remains the primary model-selection metric.

In [ ]:
metric_columns = [
    'ROC-AUC',
    'PR-AUC',
    'Accuracy',
    'Precision',
    'Recall',
    'F1',
    'Fit Time Seconds',
    'Scoring Time Seconds',
]

summary_rows = []

for model_name in baseline_models:
    model_results = stage6_fold_results[
        stage6_fold_results['Model'] == model_name
    ]

    row = {'Model': model_name}

    for metric in metric_columns:
        row[f'{metric} Mean'] = model_results[metric].mean()
        row[f'{metric} Std'] = model_results[metric].std(ddof=1)

    summary_rows.append(row)

stage6_baseline_results = pd.DataFrame(summary_rows)

comparison_columns = [
    'Model',
    'ROC-AUC Mean',
    'ROC-AUC Std',
    'PR-AUC Mean',
    'PR-AUC Std',
    'Precision Mean',
    'Recall Mean',
    'F1 Mean',
    'Accuracy Mean',
    'Fit Time Seconds Mean',
    'Scoring Time Seconds Mean',
]

stage6_comparison_display = (
    stage6_baseline_results[comparison_columns]
    .sort_values('ROC-AUC Mean', ascending=False)
    .reset_index(drop=True)
)

display(
    stage6_comparison_display.style.format(
        {
            column: '{:.4f}'
            for column in stage6_comparison_display.columns
            if column != 'Model'
        }
    )
)

### Major validation-metric comparison

The following plot compares the main validation metrics across the four models. It is descriptive evidence only; the shortlist rule is applied separately below.

In [ ]:
major_metric_columns = [
    'ROC-AUC Mean',
    'PR-AUC Mean',
    'Precision Mean',
    'Recall Mean',
    'F1 Mean',
    'Accuracy Mean',
]

plot_data = (
    stage6_baseline_results
    .set_index('Model')[major_metric_columns]
)

ax = plot_data.plot(
    kind='bar',
    figsize=(11, 6),
)
ax.set_title('Stage 6 Baseline Time-Aware Validation Metrics')
ax.set_ylabel('Score')
ax.set_xlabel('Model')
ax.set_ylim(0, 1)
ax.tick_params(axis='x', rotation=20)
ax.legend(
    title='Metric',
    bbox_to_anchor=(1.02, 1),
    loc='upper left',
)
plt.tight_layout()

comparison_figure_path = (
    FIGURE_DIR / 'stage6_baseline_metric_comparison.png'
)
plt.savefig(
    comparison_figure_path,
    dpi=150,
    bbox_inches='tight',
)
plt.show()

print('Saved:', comparison_figure_path)

## 13. Model-Interpretation Guidance

Use the executed evidence above when writing the final interpretation.

### Logistic Regression
A linear model estimates an additive relationship in log-odds after one-hot encoding and scaling. It may be more stable and interpretable than nonlinear models, but it cannot naturally capture complex interactions unless they are explicitly represented.

### Decision Tree
A single tree can represent nonlinear thresholds and interactions, but an unrestricted baseline tree can overfit. Compare its validation performance and fold stability rather than assuming a large tree is better.

### Random Forest
Random Forest averages many independently randomized trees. This bagging process usually reduces the high variance of a single tree, at the cost of more computation and less direct interpretability.

### XGBoost
XGBoost builds boosted trees sequentially, with later trees correcting earlier errors. It can capture complex nonlinear patterns, but its training/tuning process is more complex and it introduces an additional project dependency.

### Runtime and metric trade-offs
A model with slightly higher discrimination may also be slower or harder to deploy. Stage 7 selection should therefore use the pre-agreed metric hierarchy rather than relying on one result in isolation.

> **Interpret the executed results here before final submission. Do not write a model-performance conclusion until this notebook has been run successfully and the actual outputs have been reviewed.**

## 14. Stage 7 Shortlist Rule

The shortlist is determined from **training-only time-aware validation evidence**.

### Primary rule
Rank models by mean five-fold chronological **ROC-AUC**.

### Practical closeness rule
If candidates are within approximately **0.005 ROC-AUC** of the best model, treat them as practically close **for this project** rather than claiming that the small numerical difference is universally significant.

For practically close candidates, inspect:

- PR-AUC;
- F1;
- recall;
- precision;
- ROC-AUC fold stability;
- runtime;
- interpretability;
- deployment complexity.

The code below selects approximately the top two models without hard-coding any algorithm as a winner.

In [ ]:
PRACTICAL_ROC_AUC_GAP = 0.005

selection_table = (
    stage6_baseline_results[
        [
            'Model',
            'ROC-AUC Mean',
            'ROC-AUC Std',
            'PR-AUC Mean',
            'F1 Mean',
            'Recall Mean',
            'Precision Mean',
            'Fit Time Seconds Mean',
            'Scoring Time Seconds Mean',
        ]
    ]
    .sort_values('ROC-AUC Mean', ascending=False)
    .reset_index(drop=True)
)

best_roc_auc = selection_table.loc[0, 'ROC-AUC Mean']

selection_table['ROC-AUC Gap from Best'] = (
    best_roc_auc - selection_table['ROC-AUC Mean']
)

selection_table['Within 0.005 of Best'] = (
    selection_table['ROC-AUC Gap from Best']
    <= PRACTICAL_ROC_AUC_GAP
)

practical_notes = pd.DataFrame(
    {
        'Model': [
            'Logistic Regression',
            'Decision Tree',
            'Random Forest',
            'XGBoost',
        ],
        'Interpretability': [
            'High',
            'High for a single tree',
            'Moderate',
            'Moderate to lower',
        ],
        'Deployment Complexity': [
            'Low',
            'Low',
            'Moderate',
            'Moderate; extra xgboost dependency',
        ],
    }
)

selection_table = selection_table.merge(
    practical_notes,
    on='Model',
    how='left',
)

display(
    selection_table.style.format(
        {
            'ROC-AUC Mean': '{:.4f}',
            'ROC-AUC Std': '{:.4f}',
            'PR-AUC Mean': '{:.4f}',
            'F1 Mean': '{:.4f}',
            'Recall Mean': '{:.4f}',
            'Precision Mean': '{:.4f}',
            'Fit Time Seconds Mean': '{:.2f}',
            'Scoring Time Seconds Mean': '{:.2f}',
            'ROC-AUC Gap from Best': '{:.4f}',
        }
    )
)

In [ ]:
close_candidates = selection_table[
    selection_table['Within 0.005 of Best']
].copy()

close_candidates = close_candidates.sort_values(
    by=[
        'PR-AUC Mean',
        'F1 Mean',
        'Recall Mean',
        'Precision Mean',
        'ROC-AUC Std',
        'Fit Time Seconds Mean',
    ],
    ascending=[
        False,
        False,
        False,
        False,
        True,
        True,
    ],
)

selected_models = close_candidates['Model'].tolist()[:2]

if len(selected_models) < 2:
    for model_name in selection_table['Model']:
        if model_name not in selected_models:
            selected_models.append(model_name)
        if len(selected_models) == 2:
            break

stage7_shortlist = selection_table[
    selection_table['Model'].isin(selected_models)
].copy()

stage7_shortlist['_order'] = stage7_shortlist['Model'].map(
    {name: i for i, name in enumerate(selected_models)}
)
stage7_shortlist = (
    stage7_shortlist
    .sort_values('_order')
    .drop(columns='_order')
    .reset_index(drop=True)
)

print('Stage 7 shortlist determined from executed Stage 6 evidence:')
for i, model_name in enumerate(selected_models, start=1):
    print(f'{i}. {model_name}')

display(stage7_shortlist)

## 15. Save Stage 6 Experiment Evidence

The generated processed datasets are not re-saved here.

Notebook 03 saves only modelling evidence that should support reproducibility, the technical report, and Progress Evaluation 2:

- baseline summary results;
- per-fold results;
- time-aware validation out-of-fold predictions;
- fold-definition summary;
- Stage 7 shortlist;
- confusion-matrix figures;
- baseline metric-comparison figure.

In [ ]:
MODELING_REPORT_DIR = Path('reports/modeling')
MODELING_REPORT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

stage6_baseline_results.to_csv(
    MODELING_REPORT_DIR / 'stage6_baseline_results.csv',
    index=False,
)

stage6_fold_results.to_csv(
    MODELING_REPORT_DIR / 'stage6_fold_results.csv',
    index=False,
)

stage6_oof_predictions.to_csv(
    MODELING_REPORT_DIR / 'stage6_oof_predictions.csv',
    index=False,
)

fold_summary.to_csv(
    MODELING_REPORT_DIR / 'stage6_cv_fold_summary.csv',
    index=False,
)

stage7_shortlist.to_csv(
    MODELING_REPORT_DIR / 'stage6_stage7_shortlist.csv',
    index=False,
)

print('Saved Stage 6 modelling evidence:')
for path in sorted(MODELING_REPORT_DIR.glob('stage6_*')):
    print(' -', path)

print('\nSaved Stage 6 figures:')
for path in sorted(FIGURE_DIR.glob('stage6_*')):
    print(' -', path)

## 16. Stage 6 Conclusion and Handoff to Stage 7

The code below produces the factual Stage 6 handoff from the **executed results**.

The final chronological test set has still not been loaded or evaluated.

Stage 7 must use only the shortlisted validation evidence for optimization. Hyperparameters, any class-weight experiments, any justified feature ablation, and any decision-threshold selection must all remain training-only until the final model decision is frozen.

In [ ]:
ranked = (
    stage6_comparison_display
    .sort_values('ROC-AUC Mean', ascending=False)
    .reset_index(drop=True)
)

print('=' * 80)
print('STAGE 6 COMPLETION SUMMARY')
print('=' * 80)

print('\nBaseline models compared:')
for model_name in baseline_models:
    print(' -', model_name)

print('\nPrimary validation metric: mean time-aware CV ROC-AUC')

print('\nExecuted baseline ranking by mean ROC-AUC:')
for rank, row in ranked.iterrows():
    print(
        f"{rank + 1}. {row['Model']}: "
        f"ROC-AUC={row['ROC-AUC Mean']:.4f} "
        f"(SD={row['ROC-AUC Std']:.4f}), "
        f"PR-AUC={row['PR-AUC Mean']:.4f}, "
        f"F1={row['F1 Mean']:.4f}"
    )

print('\nStage 7 shortlist:')
for model_name in selected_models:
    print(' -', model_name)

print(
    '\nShortlist rule: highest mean chronological CV ROC-AUC; '
    'models within 0.005 of the best are treated as practically close '
    'for this project and secondary metrics/stability/runtime are considered.'
)

print(
    '\nLimitation: these are historical, training-only validation results. '
    'They do not guarantee performance on later unseen bookings.'
)

print('\nFINAL TEST DATA USED IN NOTEBOOK 03: NO')
print(
    'Next step: review and approve these actual Stage 6 results before '
    'creating Notebook 04 for Stage 7 optimization and final evaluation.'
)

### Final interpretation placeholder

Before committing the executed notebook, add a short written interpretation here using the actual outputs above. At minimum, record:

- which baseline achieved the highest mean chronological CV ROC-AUC;
- whether another model was within 0.005 ROC-AUC;
- the important PR-AUC / precision / recall / F1 trade-offs;
- whether any model showed noticeably higher fold-to-fold variability;
- major runtime differences;
- the two models shortlisted for Stage 7 and why;
- confirmation that the final test set was not used.

**Do not add conclusions that are not supported by the executed results.**

---

## Stage 6 complete

Notebook 03 ends here.  
**Do not perform final-test evaluation in this notebook.**